In [1]:
import gtdynamics as gtd
import numpy as np
from gtdynamics import ContactGoal, PointOnLink, Slice, Interval
from gtsam import Pose3, Point3
from gtsam.noiseModel import Isotropic, Unit
import gtsam

In [2]:
robot = gtd.CreateRobotFromFile(gtd.URDF_PATH + "/monoped.urdf")

In [3]:
# Noise models.
sigma_dynamics = 1e-5    # std of dynamics constraints.
sigma_objectives = 1e-6  # std of additional objectives.
sigma_joints = 1.85e-4   # 1.85e-4

dynamics_model_6 = Isotropic.Sigma(6, sigma_dynamics)
dynamics_model_1 = Isotropic.Sigma(1, sigma_dynamics)
dynamics_model_1_2 = Isotropic.Sigma(1, sigma_joints)
objectives_model_6 = Isotropic.Sigma(6, sigma_objectives)
objectives_model_1 = Isotropic.Sigma(1, sigma_objectives)


In [4]:
# Env parameters.
gravity, mu = np.array([0, 0, -9.8]), 1.0

opt = gtd.OptimizerSetting(sigma_dynamics)
graph_builder = gtd.DynamicsGraph(opt, gravity, None)

In [5]:
link_names = [(link.id(), link.name()) for link in robot.links()]
link_names.sort()
print(link_names)

[(0, 'base_link'), (1, 'upper_leg'), (2, 'lower_leg')]


In [6]:
import numpy as np
import gtdynamics as gtd
from gtsam import Point3

# -- Contact Point Definition --
contact_link_obj = robot.link("lower_leg")
contact_in_com_np = np.array([0, 0, -0.19])
contact_point = Point3(contact_in_com_np)
point_on_link = gtd.PointOnLink(contact_link_obj, contact_point)

# -- Phase Durations --
phase_durations = [15, 25]  # Stance has 15 steps, Flight has 25 steps

# -- Transition Contact Points --
# The list of contacts active BETWEEN phases.
# For a 2-phase hop, there is one transition (stance->flight) where the foot lifts off.
# At that moment, there are no contacts.
transition_contacts = [[]] 

# -- Other Parameters --
mu = 1.0  # Coefficient of friction

# 2. Manually Build the Transition Graphs

graph_builder = gtd.DynamicsGraph()
transition_graphs = []
cumulative_steps = 0

# Loop through each transition period (we only have one)
for i in range(len(transition_contacts)):
    # The transition happens at the end of the current phase
    cumulative_steps += phase_durations[i]
    transition_timestep = cumulative_steps - 1

    # Use dynamicsFactorGraph to build a graph for this specific transition moment.
    # This method DOES accept contact points.
    t_graph = graph_builder.dynamicsFactorGraph(
        robot,
        transition_timestep,
        transition_contacts[i],
        mu
    )
    transition_graphs.append(t_graph)

# build a multiPhaseTrajectoryFG using transition graphs
graph = graph_builder.multiPhaseTrajectoryFG(
    robot,
    phase_durations,
    transition_graphs
)

print("Successfully created the multi-phase factor graph!")
print(f"Graph size: {graph.size()} factors.")
print(f"Number of transition graphs provided: {len(transition_graphs)}")

Successfully created the multi-phase factor graph!
Graph size: 693 factors.
Number of transition graphs provided: 1


In [ ]:
# Total number of timesteps in the trajectory.
total_timesteps = sum(phase_durations)

# Define noise models for our objectives.
prior_model_l = gtsam.noiseModel.Isotropic.Sigma(6, 1e-5) # For 6D link priors
prior_model_j = gtsam.noiseModel.Isotropic.Sigma(1, 1e-5) # For 1D joint priors
goal_model = gtsam.noiseModel.Isotropic.Sigma(3, 1e-3)   # For 3D point goals
torque_model = gtsam.noiseModel.Isotropic.Sigma(1, 1.0)  # For min torque

# Get the specific links and joints we'll be constraining.
base_link = robot.link("base_link")
foot_link = robot.link("lower_leg")
joints = robot.joints()


for t in range(total_timesteps):
    # Add Minimum Torque Factors
    for joint in joints:
        graph.add(gtd.MinTorqueFactor(gtd.TorqueKey(joint.id(), t), torque_model))

    # Add Base Motion Goal using the correct addPriorPose3 method
    target_x = 0.1 * (t / total_timesteps)
    base_goal_pose = Pose3(gtsam.Rot3(), Point3(target_x, 0, 0.3))
    graph.addPriorPose3(gtd.PoseKey(base_link.id(), t), base_goal_pose, prior_model_l)

# boundry condition
final_timestep = total_timesteps - 1
# -- Initial State (t=0) --
for joint in joints:
    # Use addPriorDouble for joint angles and velocities
    graph.addPriorDouble(gtd.JointAngleKey(joint.id(), 0), 0.0, prior_model_j)
    graph.addPriorDouble(gtd.JointVelKey(joint.id(), 0), 0.0, prior_model_j)
for link in robot.links():
    # Use addPriorVector for the 6D Twist
    graph.addPriorVector(gtd.TwistKey(link.id(), 0), np.zeros(6), prior_model_l)
# -- Final State (t = final_timestep) --
for joint in joints:
    graph.addPriorDouble(gtd.JointVelKey(joint.id(), final_timestep), 0.0, prior_model_j)
for link in robot.links():
    graph.addPriorVector(gtd.TwistKey(link.id(), final_timestep), np.zeros(6), prior_model_l)

# Add Stance Foot Constraint
stance_duration = phase_durations[0]
for t in range(stance_duration):
    graph.add(gtd.PointGoalFactor(
        gtd.PoseKey(foot_link.id(), t),
        goal_model,
        contact_point,      # Point on the link (relative to its CoM)
        Point3(0, 0, 0)     # Goal position in the world frame
    ))

print(f"Graph size is now {graph.size()} factors after adding objectives.")


Graph size is now 840 factors after adding objectives.


In [ ]:
total_timesteps = sum(phase_durations)
num_phases = len(phase_durations)
# The initializer takes a list of all possible contact points.
all_contact_points = [point_on_link]

# Initialize with zero values
initializer = gtd.Initializer()
initial_values = initializer.ZeroValuesTrajectory(
    robot,
    total_timesteps,
    num_phases,
    0.0, # gaussian_noise
    all_contact_points
)

# 3. Set up and run the optimizer (this part remains the same).
params = gtsam.LevenbergMarquardtParams()
params.setVerbosityLM("SUMMARY") # Print out optimizer progress
params.setlambdaInitial(1e2)
params.setlambdaLowerBound(1e-8)
params.setlambdaUpperBound(1e10)
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_values, params)

print("\nOptimizing trajectory...")
results = optimizer.optimize()
print("\nOptimization complete!")

# get the final angle of the knee joint
knee_joint = robot.joint("knee_joint")
final_knee_angle = results.atDouble(gtd.JointAngleKey(knee_joint.id(), total_timesteps - 1))
print(f"Final Knee Angle: {final_knee_angle:.4f} radians")

RuntimeError: Attempting to retrieve value with key "24207943213776896", type stored in Values is gtsam::GenericValue<Eigen::Matrix<double, 6, 1, 0, 6, 1> > but requested type was Eigen::Matrix<double, -1, 1, 0, -1, 1>